In [1]:
import json
with open("final_merged00000000000.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# print length of data
print(f"Total items in original JSON: {len(data)}")

Total items in original JSON: 19715


In [22]:
import json

def slice_json_by_id(input_path, output_path, start_id, end_id):
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    sliced_dict = {}

    for key, value in data.items():
        item_id = value.get("id")

        # Only include if ID exists and is within the range
        if item_id is not None and start_id <= item_id <= end_id:
            sliced_dict[key] = value

    # Save the new sliced JSON
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(sliced_dict, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(sliced_dict)} items (IDs {start_id} to {end_id}) → {output_path}")


if __name__ == "__main__":
    slice_json_by_id(
        input_path="Sagar_with_ids_with_variant_count_2.json",
        output_path="slice_20000_23000.json",
        start_id=20000,
        end_id=23000
    )


Saved 1127 items (IDs 20000 to 23000) → slice_20000_23000.json


In [15]:
import json


with open("Nabin.json", "r", encoding="utf-8") as f:
    data = json.load(f)



filtered_data = {key: value for key, value in data.items() if len(value.get("variants", [])) >= 3}

# Save to a new JSON file

with open("Nabin_2.json", "w", encoding="utf-8") as f:
    json.dump(filtered_data, f, ensure_ascii=False, indent=2)

print("Filtered JSON saved to Nabin_2.json")

Filtered JSON saved to Nabin_2.json


In [27]:
import os

def merge_txt_files(file_list, output_file):
    with open(output_file, 'w', encoding='utf-8') as outfile:
        for file_path in file_list:
            with open(file_path, 'r', encoding='utf-8') as infile:
                outfile.write(infile.read())
                outfile.write("\n")  # optional
    print(f"All files merged into: {output_file}")

files = [
    "slice_345_1000.txt", "slice_1000_2000.txt", "slice_2000_4000.txt",
    "slice_4000_5000.txt", "slice_5000_6000.txt", "slice_6000_8000.txt",
    "slice_8000_10000_base.txt", "slice_10000_12000_base.txt",
    "slice_12000_14000_base.txt", "slice_14000_18000.txt", "slice_20000_23000.txt"
]

merge_txt_files(files, "merged_final.txt")


All files merged into: merged_final.txt


In [28]:
import json

def load_id_to_word_map(txt_path):
    """
    Reads the txt file and builds a mapping:
    id (int) -> word (str)
    """
    id_to_word = {}
    with open(txt_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # Expect lines like: 346,khadka
            parts = line.split(',', 1)
            if len(parts) == 2:
                id_str, word = parts
                try:
                    id_num = int(id_str.strip())
                    id_to_word[id_num] = word.strip()
                except ValueError:
                    # If first part is not a number (e.g. bad line), skip it
                    continue
            else:
                # Lines like "366" without word – ignore or handle as needed
                continue

    return id_to_word


def fill_base_words(json_input_path, json_output_path, id_to_word):
    """
    For each entry in the JSON:
    - if base_word == "....."
    - and its id exists in id_to_word mapping
      → replace base_word with the mapped word.
    """
    with open(json_input_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    for key, obj in data.items():
        # Only touch if base_word is exactly "....."
        if obj.get("base_word") == ".....":
            entry_id = obj.get("id")
            if isinstance(entry_id, int) and entry_id in id_to_word:
                obj["base_word"] = id_to_word[entry_id]

    with open(json_output_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"Updated JSON saved to {json_output_path}")


# ==== Usage ====

# 1) Path to your txt file (id,word)
txt_path = "merged_final.txt"   # change to your actual txt file

# 2) Path to your input JSON and output JSON
json_input_path = "for_merge.json"    # your original json
json_output_path = "sagar_output_filled.json"  # new json with base_word filled

# Build mapping from txt
id_to_word = load_id_to_word_map(txt_path)

# Fill base_word in JSON using the mapping
fill_base_words(json_input_path, json_output_path, id_to_word)


Updated JSON saved to sagar_output_filled.json


In [29]:
import json

def filter_json_by_id(input_path, output_path, max_id=22995):
    # Load JSON
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    new_data = {}

    # Keep only items where id <= max_id
    for key, obj in data.items():
        if isinstance(obj, dict) and "id" in obj:
            if obj["id"] <= max_id:
                new_data[key] = obj

    # Save filtered result
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(new_data, f, ensure_ascii=False, indent=2)

    print(f"Filtered JSON saved to {output_path}.")
    print(f"Original count: {len(data)}, New count: {len(new_data)}")


# Usage
filter_json_by_id("sagar_output_filled.json", "filtered_output.json")


Filtered JSON saved to filtered_output.json.
Original count: 13323, New count: 11766


In [ ]:
import json

def remove_empty_basewords(input_path, output_path):
    # Load JSON
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    new_data = {}

    # Keep only entries where base_word is NOT "....."
    for key, obj in data.items():
        if isinstance(obj, dict):
            base = obj.get("base_word", None)
            if base != ".....":
                new_data[key] = obj

    # Save the cleaned JSON
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(new_data, f, ensure_ascii=False, indent=2)

    print(f"Filtered JSON saved to {output_path}")
    print(f"Original count: {len(data)}, New count: {len(new_data)}")


# Usage
remove_empty_basewords("nabin_saksham_merged_with_ids.json", "filtered_output1111111122222.json")


Filtered JSON saved to filtered_output1111111122222.json
Original count: 11744, New count: 10934


In [2]:
import json

def merge_json_dicts(file1, file2, output_file):
    # Load first JSON
    with open(file1, "r", encoding="utf-8") as f:
        data1 = json.load(f)

    # Load second JSON
    with open(file2, "r", encoding="utf-8") as f:
        data2 = json.load(f)

    # Merge dictionaries (later file overrides if key duplicates)
    merged = {}
    merged.update(data1)
    merged.update(data2)

    # Save merged file
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(merged, f, ensure_ascii=False, indent=2)

    print(f"Merged file saved as {output_file}")

merge_json_dicts("nabin_saksham_merged_with_ids.json", "filtered_output1111111122222.json", "final_merged00000000000.json")

Merged file saved as final_merged00000000000.json


In [1]:
import json

def renumber_ids(input_path, output_path):
    # Load JSON
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Enumerate items and assign new IDs (starting from 1)
    new_data = {}
    new_id = 1
    for key, value in data.items():
        value["id"] = new_id
        new_data[key] = value
        new_id += 1

    # Save new JSON
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(new_data, f, ensure_ascii=False, indent=2)

    print(f"Done! Assigned {new_id - 1} new sequential IDs.")

# Example usage
renumber_ids("final_merged0000.json", "final_merged000000000000.json")


Done! Assigned 19715 new sequential IDs.
